# Experiments Layer - Complete Feature Showcase

This notebook demonstrates all features of the high-level experiments layer.

The experiments layer provides:
- **ExperimentConfig**: Define prompt x model matrices
- **ExperimentRunner**: Execute full experiment matrices
- **ExperimentResult**: DataFrame output with CSV persistence
- **Resume capability**: Continue interrupted experiments
- **Scheduling controls**: Concurrency and rate limiting
- **Retry options**: Configurable retry policies

## 1. Setup & Imports

Import all public API items from the inference.experiments package.

In [ ]:
# Import all experiments API items
from inference.experiments import (
    ExperimentConfig,              # Configuration for experiment runs
    ExperimentRunner,              # Executes experiment matrices
    ExperimentResult,              # Result with DataFrame and summary
    ExperimentRetryOptions,        # Retry policy configuration
    ExperimentSchedulingOptions,   # Concurrency controls
    ExperimentSummary,             # Aggregate statistics
    build_dataframe_from_csv,      # Load experiment CSV into DataFrame
)

# Also need the base inference client
from inference import create_client

import asyncio
import pandas as pd
from pathlib import Path

print("✓ All experiments API items imported successfully")

## 2. Basic Experiment Matrix

Create and run a simple experiment matrix with multiple prompts and models.

**ExperimentConfig parameters:**
- `experiment_name`: Labels outputs and creates log directory
- `model_aliases`: List of model aliases to test (columns)
- `prompts`: List of prompts to test (rows)
- `retry`: `ExperimentRetryOptions` for retry behavior
- `scheduling`: `ExperimentSchedulingOptions` for concurrency

In [ ]:
async def basic_experiment_demo():
    """Run a simple 3x1 experiment matrix."""
    
    # Create the inference client
    client = create_client("config/inference.example.yaml")
    print("✓ Client created\n")
    
    # Define experiment configuration
    config = ExperimentConfig(
        experiment_name="basic-demo",
        model_aliases=["mock-test"],  # Single model for demo
        prompts=[
            "What is 2+2?",
            "What is the capital of France?",
            "Name a primary color.",
        ],
    )
    
    print("Experiment Configuration:")
    print(f"  Name: {config.experiment_name}")
    print(f"  Models: {config.model_aliases}")
    print(f"  Prompts: {len(config.prompts)}")
    print(f"  Matrix size: {len(config.prompts)} x {len(config.model_aliases)} = {len(config.prompts) * len(config.model_aliases)} cells\n")
    
    # Create runner and execute
    runner = ExperimentRunner(client)
    print("Running experiment...")
    result = await runner.run(config)
    
    print(f"\n✓ Experiment complete!")
    print(f"  CSV saved to: {result.csv_path}")
    print(f"  CSV name: {result.csv_name}")
    
    return result

# Run the basic experiment
basic_result = await basic_experiment_demo()

## 3. ExperimentResult & Summary

Inspect the experiment results including the DataFrame and summary statistics.

**ExperimentResult fields:**
- `dataframe`: Pandas DataFrame with the full matrix
- `csv_path`: Path to the saved CSV file
- `csv_name`: Filename of the CSV
- `summary`: `ExperimentSummary` with aggregate stats

**ExperimentSummary fields:**
- `prompt_count`: Number of prompts
- `model_count`: Number of models
- `total_cells`: Total cells in matrix
- `completed_cells`: Successfully completed cells
- `failed_cells`: Failed cells
- `rate_limited_cells`: Rate-limited cells

In [ ]:
def inspect_experiment_result(result):
    """Inspect the experiment result and summary."""
    
    print("=" * 60)
    print("EXPERIMENT RESULT")
    print("=" * 60)
    
    # Show summary statistics
    summary = result.summary
    print("\nSummary Statistics:")
    print(f"  Prompts: {summary.prompt_count}")
    print(f"  Models: {summary.model_count}")
    print(f"  Total Cells: {summary.total_cells}")
    print(f"  Completed: {summary.completed_cells}")
    print(f"  Failed: {summary.failed_cells}")
    print(f"  Rate Limited: {summary.rate_limited_cells}")
    
    # Show the DataFrame
    print("\n" + "=" * 60)
    print("DATAFRAME")
    print("=" * 60)
    print(result.dataframe)
    
    # Show DataFrame info
    print("\n" + "=" * 60)
    print("DATAFRAME INFO")
    print("=" * 60)
    print(f"  Shape: {result.dataframe.shape}")
    print(f"  Columns: {list(result.dataframe.columns)}")
    print(f"  Index: {result.dataframe.index.name}")

# Inspect the result from the previous cell
inspect_experiment_result(basic_result)

## 4. Resume from CSV

Resume an interrupted experiment using `resume_from_existing_csv`.

**Resume behavior:**
- Set `resume_from_existing_csv=True` to enable resume
- Automatically finds the latest CSV in `logs/<experiment-name>/`
- Skips already completed cells
- Processes only remaining cells

**Options:**
- `existing_csv_path`: Explicitly specify CSV path (optional)

In [ ]:
async def resume_experiment_demo():
    """Demonstrate resuming from an existing CSV."""
    
    client = create_client("config/inference.example.yaml")
    
    # Configuration with resume enabled
    config = ExperimentConfig(
        experiment_name="basic-demo",  # Same name as before
        model_aliases=["mock-test"],
        prompts=[
            "What is 2+2?",
            "What is the capital of France?",
            "Name a primary color.",
            "What is the speed of light?",  # New prompt
        ],
        resume_from_existing_csv=True,  # Enable resume
        # existing_csv_path=Path("..."),  # Optional: explicit path
    )
    
    print("Resuming experiment with resume_from_existing_csv=True")
    print(f"  Experiment: {config.experiment_name}")
    print(f"  Total prompts: {len(config.prompts)}")
    print("\nNote: Already completed cells will be skipped\n")
    
    runner = ExperimentRunner(client)
    result = await runner.run(config)
    
    print(f"\n✓ Resumed experiment complete!")
    print(f"  Total cells: {result.summary.total_cells}")
    print(f"  Completed: {result.summary.completed_cells}")
    
    return result

# Run resume demo
resume_result = await resume_experiment_demo()

## 5. Extending Experiments

Add new prompts to an existing experiment.

When you add new prompts to an experiment with `resume_from_existing_csv=True`:
- Existing prompts are preserved
- New prompts are added to the matrix
- Only new cells are processed

This allows incremental experiment expansion.

In [ ]:
async def extend_experiment_demo():
    """Demonstrate extending an experiment with new prompts."""
    
    client = create_client("config/inference.example.yaml")
    
    # Add more prompts to the existing experiment
    config = ExperimentConfig(
        experiment_name="basic-demo",
        model_aliases=["mock-test"],
        prompts=[
            "What is 2+2?",
            "What is the capital of France?",
            "Name a primary color.",
            "What is the speed of light?",
            "Who wrote Romeo and Juliet?",  # Additional prompt
            "What is the largest planet?",    # Additional prompt
        ],
        resume_from_existing_csv=True,
    )
    
    print("Extending experiment with new prompts")
    print(f"  Previous prompts: 4")
    print(f"  New total: {len(config.prompts)}")
    print(f"  New prompts to process: 2\n")
    
    runner = ExperimentRunner(client)
    result = await runner.run(config)
    
    print(f"\n✓ Extended experiment complete!")
    print(f"  Total cells: {result.summary.total_cells}")
    print(f"  DataFrame shape: {result.dataframe.shape}")
    
    return result

# Run extension demo
extended_result = await extend_experiment_demo()

## 6. Analyzing Results

Use `build_dataframe_from_csv` to load and analyze experiment CSVs.

This function:
- Loads any experiment CSV file
- Returns a Pandas DataFrame
- Preserves all metadata and responses

Useful for post-hoc analysis and visualization.

In [ ]:
def analyze_results_demo():
    """Demonstrate analyzing experiment results from CSV."""
    
    # Load the CSV from the previous experiment
    csv_path = extended_result.csv_path
    
    print(f"Loading CSV: {csv_path}\n")
    
    # Build DataFrame from CSV
    df = build_dataframe_from_csv(csv_path)
    
    print("=" * 60)
    print("LOADED DATAFRAME")
    print("=" * 60)
    print(df)
    
    # Inspect individual cells
    print("\n" + "=" * 60)
    print("CELL INSPECTION")
    print("=" * 60)
    
    # DataFrame columns are: ["prompt_id", "prompt", *model_aliases]
    first_prompt_id = df.iloc[0]["prompt_id"]
    first_prompt_text = df.iloc[0]["prompt"]
    model_alias = df.columns[2]  # First model column (after prompt_id, prompt)
    cell_data = df.iloc[0][model_alias]
    
    print(f"\nFirst prompt_id: {first_prompt_id[:40]}...")
    print(f"Prompt text: {first_prompt_text[:50]}...")
    print(f"Model alias: {model_alias}")
    print(f"Cell data: {str(cell_data)[:100]}...")
    
    # Show DataFrame statistics
    print("\n" + "=" * 60)
    print("DATAFRAME STATISTICS")
    print("=" * 60)
    print(f"  Shape: {df.shape}")
    print(f"  Memory usage: {df.memory_usage(deep=True).sum()} bytes")
    print(f"  Non-null cells: {df.notna().sum().sum()}")
    print(f"  Null cells: {df.isna().sum().sum()}")

# Run analysis demo
analyze_results_demo()

## 7. Scheduling Controls

Configure concurrency with `ExperimentSchedulingOptions`.

**Scheduling options:**
- `max_concurrency`: Global concurrent request limit (0 = unlimited)
- `per_model_concurrency`: Per-model concurrent limit (0 = unlimited)
- `interleave_model_aliases`: If True, interleave models; if False, group by model
- `max_retry_after_wait_seconds`: Max wait time for rate limit retries

In [ ]:
async def scheduling_demo():
    """Demonstrate scheduling controls."""
    
    client = create_client("config/inference.example.yaml")
    
    # Configure scheduling options
    scheduling = ExperimentSchedulingOptions(
        max_concurrency=2,              # Max 2 concurrent requests globally
        per_model_concurrency=1,        # Max 1 concurrent per model
        interleave_model_aliases=True,  # Interleave models for fairness
        max_retry_after_wait_seconds=300,  # Max 5 min wait for rate limits
    )
    
    print("Scheduling Configuration:")
    print(f"  max_concurrency: {scheduling.max_concurrency}")
    print(f"  per_model_concurrency: {scheduling.per_model_concurrency}")
    print(f"  interleave_model_aliases: {scheduling.interleave_model_aliases}")
    print(f"  max_retry_after_wait_seconds: {scheduling.max_retry_after_wait_seconds}\n")
    
    config = ExperimentConfig(
        experiment_name="scheduling-demo",
        model_aliases=["mock-test"],
        prompts=[
            "Prompt 1",
            "Prompt 2",
            "Prompt 3",
        ],
        scheduling=scheduling,
    )
    
    runner = ExperimentRunner(client)
    print("Running with scheduling controls...\n")
    result = await runner.run(config)
    
    print(f"\n✓ Complete!")
    print(f"  Processed {result.summary.total_cells} cells")
    
    return result

# Run scheduling demo
scheduling_result = await scheduling_demo()

## 8. Retry Options

Configure retry behavior with `ExperimentRetryOptions`.

**Retry options:**
- `max_retries`: Maximum retry attempts per cell
- `base_delay`: Initial delay between retries (seconds)
- `max_delay`: Maximum delay between retries (seconds)

Uses exponential backoff: delay = min(base_delay * 2^attempt, max_delay)

In [ ]:
async def retry_options_demo():
    """Demonstrate retry configuration."""
    
    client = create_client("config/inference.example.yaml")
    
    # Configure retry options
    retry = ExperimentRetryOptions(
        max_retries=3,      # Retry up to 3 times
        base_delay=1.0,     # Start with 1 second delay
        max_delay=30.0,     # Cap at 30 seconds
    )
    
    print("Retry Configuration:")
    print(f"  max_retries: {retry.max_retries}")
    print(f"  base_delay: {retry.base_delay}s")
    print(f"  max_delay: {retry.max_delay}s")
    print("\nBackoff sequence would be:")
    for i in range(retry.max_retries):
        delay = min(retry.base_delay * (2 ** i), retry.max_delay)
        print(f"  Attempt {i+1}: {delay}s delay")
    print()
    
    config = ExperimentConfig(
        experiment_name="retry-demo",
        model_aliases=["mock-test"],
        prompts=["Test prompt"],
        retry=retry,
    )
    
    runner = ExperimentRunner(client)
    result = await runner.run(config)
    
    print(f"✓ Complete with retry policy!")
    print(f"  Retries used: {result.summary.failed_cells == 0}")
    
    return result

# Run retry demo
retry_result = await retry_options_demo()

## 9. Multi-Model Comparison

Run experiments with multiple model aliases for comparison.

The matrix will have:
- Rows: Prompts
- Columns: Model aliases
- Cells: Responses from each model for each prompt

**Note:** Model aliases must be unique. The same provider can be configured under different aliases.

In [ ]:
async def multi_model_demo():
    """Demonstrate multi-model comparison experiment."""
    
    client = create_client("config/inference.example.yaml")
    
    # Note: mock-test-a and mock-test-b both resolve to mock-test provider
    # In real usage, use different aliases like ["gpt-4o", "claude-3-5-sonnet"]
    config = ExperimentConfig(
        experiment_name="multi-model-demo",
        model_aliases=[
            "mock-test-a",
            "mock-test-b",  # Both resolve to mock-test provider
        ],
        prompts=[
            "What is machine learning?",
            "Explain neural networks.",
        ],
    )
    
    print("Multi-Model Experiment:")
    print(f"  Models: {len(config.model_aliases)}")
    print(f"  Prompts: {len(config.prompts)}")
    print(f"  Matrix: {len(config.prompts)} x {len(config.model_aliases)} = {len(config.prompts) * len(config.model_aliases)} cells\n")
    
    runner = ExperimentRunner(client)
    result = await runner.run(config)
    
    print(f"\n✓ Multi-model experiment complete!")
    print(f"\nDataFrame:")
    print(result.dataframe)
    
    return result

# Run multi-model demo
multi_model_result = await multi_model_demo()

## Summary

This notebook demonstrated all experiments layer features:

| Feature | Class/Function | Purpose |
|---------|---------------|---------|
| Configuration | `ExperimentConfig` | Define experiment matrix |
| Runner | `ExperimentRunner` | Execute experiments |
| Result | `ExperimentResult` | DataFrame + CSV + summary |
| Summary | `ExperimentSummary` | Aggregate statistics |
| Retry Options | `ExperimentRetryOptions` | Configure retry behavior |
| Scheduling | `ExperimentSchedulingOptions` | Concurrency controls |
| Resume | `resume_from_existing_csv` | Continue interrupted runs |
| Analysis | `build_dataframe_from_csv` | Load CSV for analysis |

### Next Steps

- See `inference_example.ipynb` for low-level inference features
- See `docs/inference-usage.md` for detailed API documentation
- See `examples/run_experiment_matrix.py` for script-based experiments